# Módulo outliers
Regla de Tukey (IQR) **por estrato** (materia homologada × ámbito): comparar un juzgado civil
con uno penal no tiene sentido. Ninguna fila se elimina: se marcan banderas, se acota la cola
al percentil 95 dentro del estrato (winsorización) y se agrega log(1 + x) a los volúmenes.

In [ ]:
import numpy as np
import pandas as pd


def calcular_estadisticas_iqr(serie, factor=1.5, factor_severo=3.0):
    s = serie.dropna()
    if len(s) < 4:
        return {"n": len(s), "q1": np.nan, "mediana": np.nan, "q3": np.nan, "iqr": np.nan,
                "limite_inf": np.nan, "limite_sup": np.nan, "limite_severo": np.nan, "p95": np.nan}
    q1 = float(s.quantile(0.25))
    mediana = float(s.median())
    q3 = float(s.quantile(0.75))
    p95 = float(s.quantile(0.95))
    iqr = q3 - q1
    # Con IQR = 0 (casi todos los valores iguales) el límite de Tukey marcaría todo lo distinto.
    if iqr == 0:
        lim_sup = max(q3 + 1.0, p95)
        lim_severo = max(q3 + 2.0, p95 * 1.5)
    else:
        lim_sup = q3 + factor * iqr
        lim_severo = q3 + factor_severo * iqr
    lim_inf = max(0.0, q1 - factor * iqr)
    return {"n": len(s), "q1": q1, "mediana": mediana, "q3": q3, "iqr": iqr,
            "limite_inf": lim_inf, "limite_sup": lim_sup, "limite_severo": lim_severo, "p95": p95}


def detectar_outliers_estratificados(df, columna, columnas_estrato, filtro_mascara=None):
    es_outlier = pd.Series(False, index=df.index, dtype=bool)
    es_severo = pd.Series(False, index=df.index, dtype=bool)
    registros = []
    if filtro_mascara is None:
        sub = df
    else:
        sub = df[filtro_mascara]
    grupos = sub.groupby(columnas_estrato, observed=True).groups
    for nombre, indices in grupos.items():
        if isinstance(nombre, tuple):
            partes = []
            for g in nombre:
                partes.append(str(g))
            etiqueta = " __ ".join(partes)
        else:
            etiqueta = str(nombre)
        stats = calcular_estadisticas_iqr(sub.loc[indices, columna].dropna())
        stats["estrato"] = etiqueta
        stats["columna"] = columna
        if not np.isnan(stats["limite_sup"]):
            idx_out = indices[sub.loc[indices, columna] > stats["limite_sup"]]
            idx_sev = indices[sub.loc[indices, columna] > stats["limite_severo"]]
            es_outlier.loc[idx_out] = True
            es_severo.loc[idx_sev] = True
            stats["n_outliers"] = len(idx_out)
            stats["n_severos"] = len(idx_sev)
        else:
            stats["n_outliers"] = 0
            stats["n_severos"] = 0
        registros.append(stats)
    return es_outlier, es_severo, pd.DataFrame(registros)


def winsorizar_estratificado(df, columna, columnas_estrato, percentil=0.95, filtro_mascara=None):
    resultado = df[columna].copy().astype(float)
    if filtro_mascara is None:
        sub = df
    else:
        sub = df[filtro_mascara]
    grupos = sub.groupby(columnas_estrato, observed=True).groups
    for nombre, indices in grupos.items():
        valores = sub.loc[indices, columna].dropna()
        if len(valores) > 0:
            tope = float(valores.quantile(percentil))
            afectados = indices[sub.loc[indices, columna] > tope]
            resultado.loc[afectados] = tope
    return resultado

In [ ]:
def aplicar_transformaciones_curadas(df):
    res = df.copy()
    estrato = ["materia_homologada", "ambito"]
    es_proceso = res["tipo_elemento_analitico"] == "proceso"
    con_congestion = es_proceso & res["tasa_congestion"].notnull()
    resumenes = []

    variables = [
        ("atendidas", "carga", es_proceso),
        ("nuevas_ingresadas", "ingresos", es_proceso),
        ("tasa_congestion", "congestion", con_congestion),
        ("duracion_estimada_dias", "duracion", con_congestion),
    ]
    for columna, nombre, mascara in variables:
        out, sev, resumen = detectar_outliers_estratificados(res, columna, estrato, filtro_mascara=mascara)
        res["es_outlier_" + nombre + "_iqr"] = out
        res["es_outlier_severo_" + nombre + "_iqr"] = sev
        resumenes.append(resumen)

    res["tasa_congestion_winsorizada"] = winsorizar_estratificado(res, "tasa_congestion", estrato, percentil=0.95, filtro_mascara=con_congestion)
    res["duracion_estimada_winsorizada"] = winsorizar_estratificado(res, "duracion_estimada_dias", estrato, filtro_mascara=con_congestion)

    for c in ["atendidas", "nuevas_ingresadas", "resueltas", "pendientes_fin", "ingresos_totales"]:
        if c in res.columns:
            res["log_" + c] = np.log1p(res[c].fillna(0).astype(float).clip(lower=0))

    return res, pd.concat(resumenes, ignore_index=True)